In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [2]:
import gradio as gr
from Day4LLMCalling import Llms, tools
from openai import OpenAI
from google import genai

client = genai.Client(api_key=os.getenv('GOOGLE_API_KEY'))
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/",api_key=os.getenv('GOOGLE_API_KEY'))


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [3]:
import base64
from PIL import Image
from io import BytesIO

def artist(city):
    if not city:
        return None
    image_response = gemini.images.generate(
        prompt=f'Generate an artistic image for {city}',
        model="gemini-3.1-flash-image",
        n=1,
        size = "1024x1024",
        response_format="b64_json"
    )
    image_b64 = base64.b64decode(image_response.data[0].b64_json)
    image_result = Image.open(BytesIO(image_b64))
    return image_result



In [4]:
def talker(text):
    # Using the native Gemini 3.1 Flash TTS model
    response = client.models.generate_content(
        model="gemini-3.1-flash-tts-preview",
        contents=text
    )
    
    # The response contains the raw audio bytes
    audio_bytes = response.executable_ad_data # or response.data depending on version
    return audio_bytes

In [5]:
from Day4LLMCalling.messageSeries import mSeries


def wrapLlm(message):
    response,tool_arguments = Llms.callModel(message, source='gemini',tools=tools,return_tool_arguments=True)
    history = mSeries.promptList.get('0',{}).get('gemini-3-flash-preview',[])
    if len(tool_arguments)==0:
        return response, history, None
    return response, history, tool_arguments[0]['destination_city']

In [7]:
with gr.Blocks() as ui:
    city_state = gr.State()
    audio_state = gr.State()
    with gr.Row():
        chat_history = gr.Chatbot(height=500, show_label=False)
        image_box = gr.Image(height=500, interactive=False, show_label=False)
    with gr.Row():
        audio_box = gr.Audio(autoplay=True)
    with gr.Row():
        message_box = gr.Textbox(label='Chat with AI')

        message_box.submit(
            fn = wrapLlm,
            inputs=[message_box],
            outputs=[audio_state,chat_history,city_state]
            ).then(
            fn=artist,
            inputs=[city_state], 
            outputs=[image_box])
            # ).then(
            # fn=talker,
            # inputs=[audio_state],
            # outputs=[audio_box]
            # )

ui.launch()
        

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
